In [9]:
import sys

# !{sys.executable} -m pip uninstall torch torchvision -y
# !{sys.executable} -m pip install torch torchvision

In [10]:
# Import
import os
import cv2 # used for opening video files and extracting frames
import torch # used for loading the model and performing inference
import numpy as np # used for numerical operations
import mediapipe as mp # used for hand landmark detection
from pathlib import Path # used for handling file paths
from torch.utils.data import Dataset, DataLoader # used for creating a custom dataset and data loader
from torchvision import transforms # used for image transformations
from sklearn.model_selection import train_test_split # used for splitting the dataset into training and testing sets

In [ ]:
DATA_DIR = "./data/wlasl_top20" # path to the dataset 
NUM_FRAMES = 16 # number of frames to extract from each video
IMG_SIZE = 224 # size to which each frame will be resized
BATCH_SIZE = 8 
NUM_CLASSES = 20
RANDOM_SEED = 42

# get class labels from folder names
CLASSES = sorted(os.listdir(DATA_DIR))
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)} # create a mapping from class names to indices
print(f"class to index mapping: {CLASS_TO_IDX}")

class to index mapping: {'accident': 0, 'bar': 1, 'bed': 2, 'before': 3, 'bowling': 4, 'call': 5, 'candy': 6, 'champion': 7, 'change': 8, 'check': 9, 'cold': 10, 'computer': 11, 'cool': 12, 'cousin': 13, 'deaf': 14, 'delay': 15, 'drink': 16, 'far': 17, 'go': 18, 'help': 19}


In [14]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(static_image_mode=True, max_num_hands=2)

PADDING = 20  # pixels of padding around the hand crop

def crop_hand_region(frame):
    """Uses MediaPipe to find hands and return a cropped region. Falls back to full frame."""
    h, w = frame.shape[:2]
    results = hands_detector.process(frame)
    
    if results.multi_hand_landmarks:
        # get bounding box across all detected hand landmarks
        all_x = [lm.x for hand in results.multi_hand_landmarks for lm in hand.landmark]
        all_y = [lm.y for hand in results.multi_hand_landmarks for lm in hand.landmark]
        
        x1 = max(0, int(min(all_x) * w) - PADDING)
        y1 = max(0, int(min(all_y) * h) - PADDING)
        x2 = min(w, int(max(all_x) * w) + PADDING)
        y2 = min(h, int(max(all_y) * h) + PADDING)
        
        crop = frame[y1:y2, x1:x2]
        if crop.size > 0:
            return crop
    
    return frame  # fallback to full frame if no hand detected

def extract_frames(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = crop_hand_region(frame)  # <-- NEW: hand crop
            frames.append(frame)
    
    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1])
    
    return frames

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [ ]:
# sample video to test
sample_class = CLASSES[0]
sample_video = list(Path(f"{DATA_DIR}/{sample_class}").glob("*.mp4"))[0]

frames = extract_frames(str(sample_video))
print(f"Extracted {len(frames)} frames from {sample_video.name}")
print(f"Frame shape: {frames[0].shape}")

Extracted 16 frames from 00623.mp4
Frame shape: (240, 320, 3)


In [ ]:
# Transform frames (resize, normalize, etc.) and create a custom dataset class to load the data for training.

train_transforms = transforms.Compose([
    # conditional changes for training
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # resize frames to a 224x224 square
    
    # Augementation techniques for training (flip, rotate, clor jitter, )
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalize using ImageNet stats
])

validation_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalize using ImageNet stats
])

In [ ]:
'''
Class Methods:

__init__    -- initializes the dataset with video paths, labels, and transformations

__len__     -- tells how many samples are in the dataset

__getitem__ -- retrieves a single sample (frames and label) from the dataset given an index
                extractts 16 frames from theh video using extract_frames() function, applies the 
                transformations to each frame, and stacks them into a tensor of shape (num_frames, channels, height, width)
                ex. (16, 3, 224, 224) for 16 frames, 3 color channels, and 224x224 pixel size
'''


class ASLDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        
        frames = extract_frames(video_path)
        
        if self.transform:
            frames = [self.transform(frame) for frame in frames]
        
        # stack frames into a tensor of shape (num_frames, channels, height, width)
        frames_tensor = torch.stack(frames)
        
        return frames_tensor, label
    

In [ ]:
# Building filepaths, labels, and splits for the dataset
'''
Two main steps:
1. Loops through every class folder in wlasl_top20, finds all .mp4 files, and builds two parallel lists
    - video_paths: a list of file paths to each video
    - labels: a list of corresponding class indices for each video
    (such that index 0 of both lists always refrences the same video and label)
    
2. Then, train_test_split is used twice, the first time to split the dataset into 70% training and 300% temp. The second time, temp is split into 50% validation and 50% testing, resulting in a final split of 70% training, 15% validation, and 15% testing.
    The seond, call splits the 30% tep enenly into 15% validation and 15% testin.
    'stratify' parameter ensures each split has a proportional representation of all classes, so no classs ends up only in train or only in test
    
'''

from sklearn.model_selection import train_test_split

video_paths = []
labels = []

for cls in CLASSES:
    cls_dir = Path(DATA_DIR) / cls
    # print(f"Processing class '{cls}' with path: {cls_dir}")
    
    for video in cls_dir.glob("*.mp4"): # Get all .mp4 files in the class directory
        # append their paths to video_paths and their corresponding class indices to labels
        video_paths.append(str(video)) 
        labels.append(CLASS_TO_IDX[cls])
        
print(f"Total videos found: {len(video_paths)}")
# Split into train (70%) and temp (30%)

# WHERE:
# X_train, Y_train: training video paths and labels
# X_temp, Y_temp: temporary video paths and labels that will be further split into validation

X_train, X_temp, Y_train, Y_temp = train_test_split(
    video_paths, labels, test_size=0.30, random_state=RANDOM_SEED, stratify=labels
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=Y_temp
)   

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


Total videos found: 227
Train: 158 | Val: 34 | Test: 35


In [ ]:
'''
This cell takes teh split lists and creates 3 ASLDataset objects, 
    one each for train, val, and test
    
'''

train_dataset = ASLDataset(X_train, Y_train, transform=train_transforms)
val_dataset = ASLDataset(X_val, Y_val, transform=validation_transforms)
test_dataset = ASLDataset(X_test, Y_test, transform=validation_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False) 